<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/h1h0_shared_substrate_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A shared laminar substrate across error types — partial H1 test (Neuropixels)

Results 1–5 establish that four kinds of violated expectation each drive a positive
prediction-error (PE) response. "All positive," though, is consistent with **both** hypotheses
from the background review (`arXiv:2504.09614`):

- **H1** — one *common* deviance-detection mechanism serves every error type;
- **H0** — *separate* circuits, each specialized for its own error type.

The test that separates them: **do the same anatomical populations carry different error types?**
Five animals (830794, 830795, 830848, 830851, 830852) were recorded across the **feature-oddball**,
**sequence**, and **duration** paradigms — in separate sessions, but with probes targeting the same
visual areas — so with the CCF area/layer labels we can compare the PE's *anatomical profile* across
error types.

> **What a positive result would and wouldn't show.** These are separate sessions, so we compare
> *populations* (area × layer), not the *same units* across paradigms. A shared anatomical profile
> is evidence for a common substrate; it cannot prove single neurons are multiplexed. We report the
> result with that limit explicit.

In [ ]:
# Setup
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=True)
try:
    import pynwb, remfile, h5py, fsspec, aiohttp  # noqa
except ImportError:
    _pip("pynwb","remfile","h5py","fsspec","aiohttp","requests")
import numpy as np, pandas as pd, h5py, remfile, requests, re, time
from scipy import stats as ss
from scipy.stats import spearmanr, pearsonr
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

QUICK = True   # True: 3 of the 5 shared animals, fast Colab pass. False: all 5.
SHARED = ["830794","830848","830851"] if QUICK else ["830794","830795","830848","830851","830852"]
print(f"{len(SHARED)} shared animals: {SHARED}")

In [ ]:
# NWB streaming + CCF-corrected unit->area/layer mapping
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])

# asset ids per paradigm, restricted to the 5 shared animals (verified against the session index)
ASSETS = {
  "oddball":  {"830794":"a23eba88-5d08-4e4b-89e1-a2a10fd143b5","830795":"1a3aac78-34f5-44c4-b644-7c16bc313ceb",
               "830848":"228c2c2e-1daf-4bf6-9f66-eb6b2bce5ba5","830851":"9b9e8abe-7b43-47f1-b8e1-4114f87898a1",
               "830852":"b2f4bb1b-5092-4392-b4fa-b25802ea7769"},
  "sequence": {"830794":"6068eec9-d614-471c-aa33-0ab27dc66475","830795":"4095a089-8cb6-4101-821c-9d93e2c2dee7",
               "830848":"84b56f6e-7ade-4790-96e1-9bbc82cd5688","830851":"24822bc8-01d9-473f-9eef-615fa01119d9",
               "830852":"6f14398a-7abc-446c-a5ab-c008f52c7129"},
  "duration": {"830794":"b383b4ff-a2b9-4497-8c8f-9f77de89e1b0","830795":"c9024a36-e476-4ffc-9e12-666e6ed3aec7",
               "830848":"cc3b7c7b-d489-460c-b575-40b9edf59e2c","830851":"0167df0a-b5f1-45bd-9659-c32ac132d875",
               "830852":"3ffe2e3d-9c41-459f-819f-1b72ac5284a2"},
}

def unit_area_layer(fh):
    """CCF-corrected per-unit area+layer. extremum_channel_index is PER-PROBE."""
    U=fh["units"]; n=len(U["spike_times_index"][:])
    Eg=fh["general/extracellular_ephys/electrodes"]; eloc=col(Eg,"location"); egroup=col(Eg,"group_name")
    dev=col(U,"device_name"); eci=U["extremum_channel_index"][:]
    groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
    offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups}; bl={gn:int((egroup==gn).sum()) for gn in groups}
    def d2g(dd):
        for gn in groups:
            if dd[-1].lower() in gn.lower(): return gn
        return groups[0]
    dmap={dd:d2g(dd) for dd in set(dev)}
    return [eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)]

## Extract PE time courses for each error type (same 5 animals)

In [ ]:
# Per-paradigm PSTH extractors. Each returns per-unit PSTHs for the deviant and its matched
# control, plus area/layer. Windows match the individual Result notebooks.
def psth_maker(EDGES,PRE,POST):
    TCEN=EDGES[:-1]+(EDGES[1]-EDGES[0])/2
    def psth(sp,tt):
        if len(tt)==0: return np.full(len(TCEN),np.nan)
        lo=np.searchsorted(sp,tt.min()-PRE); hi=np.searchsorted(sp,tt.max()+POST); sp2=sp[lo:hi]
        rel=(sp2[None,:]-tt[:,None]).ravel(); rel=rel[(rel>=-PRE)&(rel<POST)]
        return np.histogram(rel,EDGES)[0]/(len(tt)*(EDGES[1]-EDGES[0]))
    return psth,TCEN

def extract(paradigm, subj):
    aid=ASSETS[paradigm][subj]
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        U=fh["units"]; st=U["spike_times"][:]; sti=U["spike_times_index"][:]; n=len(sti)
        qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        locs=unit_area_layer(fh)
        vis=[i for i in range(n) if qc[i] and re.match("VIS",str(locs[i]))]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        I=fh["intervals"]
        if paradigm=="oddball":
            g=I["Standard mismatch block_presentations"]; TT=col(g,"TrialType"); ori=col(g,"Orientation").astype(float); ts=g["start_time"][:]
            gc=I["Control block 1_presentations"]; cori=col(gc,"Orientation").astype(float); cts=gc["start_time"][:]
            cdur=gc["stop_time"][:]-gc["start_time"][:]
            dev=ts[(TT=="orientation_90")]; ctrl=cts[(np.abs(np.degrees(cori)-90)<5)&(cdur>=0.3)]
            EDGES=np.arange(-0.1,0.5,0.01)
        elif paradigm=="sequence":
            g=I["Sequence mismatch block_presentations"]; TT=col(g,"TrialType"); tis=col(g,"TrialInSequence").astype(float); ts=g["start_time"][:]
            gc=I["Control block 2_presentations"]; cori=col(gc,"Orientation").astype(float); ctf=col(gc,"TemporalFrequency").astype(float)
            cts=gc["start_time"][:]; cdur=gc["stop_time"][:]-gc["start_time"][:]
            dev=ts[(TT=="orientation_90")&(tis==3)]; ctrl=cts[(np.abs(np.degrees(cori)-90)<5)&(np.abs(ctf-2)<0.1)&(cdur>=0.2)]
            EDGES=np.arange(-0.3,0.5,0.02)
        else:  # duration: omission at expected time vs its own baseline
            g=I["Duration mismatch block_presentations"]; TT=col(g,"TrialType"); ts=g["start_time"][:]
            dev=ts[TT=="omission"]; ctrl=None
            EDGES=np.arange(-0.3,0.6,0.02)
        psth,TCEN=psth_maker(EDGES,-EDGES[0],EDGES[-1])
        rows=[];PD=[];PC=[]
        for u in vis:
            sp=sp_(u); m=re.match(r"(VIS[a-z]*)(\d[a-b]?)?",str(locs[u]))
            rows.append((subj,paradigm,m.group(1) if m else str(locs[u]),(m.group(2) or "") if m else ""))
            PD.append(psth(sp,dev) if len(dev)>=5 else np.full(len(TCEN),np.nan))
            PC.append(psth(sp,ctrl) if (ctrl is not None and len(ctrl)>=5) else np.full(len(TCEN),np.nan))
        meta=pd.DataFrame(rows,columns=["subject","paradigm","area","layer"])
        return meta,np.array(PD),np.array(PC),TCEN
    finally: fh.close()

In [ ]:
# Run all three paradigms across the shared animals
DATA={}; t0=time.time()
for para in ["oddball","sequence","duration"]:
    metas=[];PDs=[];PCs=[];TC=None
    for s in SHARED:
        meta,PD,PC,TC=extract(para,s); metas.append(meta); PDs.append(PD); PCs.append(PC)
        print(f"  {para}/{s}: {len(meta)} units ({time.time()-t0:.0f}s)")
    DATA[para]=(pd.concat(metas,ignore_index=True),np.vstack(PDs),np.vstack(PCs),TC)
print("done")

## Prediction-error time courses and the H1/H0 anatomical test

In [ ]:
# PE per unit: oddball/sequence = (dev - control), each baseline-subtracted (t<0), in Hz.
#              duration = omission response at expected time, own far-baseline (-250,-100 ms) subtracted.
areas=["VISp","VISl","VISlm","VISa"]
PE={}
for para,(meta,PD,PC,TC) in DATA.items():
    tpre=TC<0
    if para=="duration":
        far=(TC>=-0.25)&(TC<-0.1)
        pe=PD-np.nanmean(PD[:,far],1,keepdims=True)
    else:
        pe=(PD-np.nanmean(PD[:,tpre],1,keepdims=True))-(PC-np.nanmean(PC[:,tpre],1,keepdims=True))
    PE[para]=(meta,pe,TC)

def area_traces(meta,pe,TC):
    out={}
    for a in areas:
        sel=(meta.area==a).values
        if sel.sum()>=8:
            arr=pe[sel]; out[a]=(np.nanmean(arr,0),np.nanstd(arr,0)/np.sqrt(np.sum(~np.isnan(arr[:,0]))),int(sel.sum()))
    return out

# Figure 1 — PE time courses by area
acol={"VISp":"#1b4079","VISl":"#c0392b","VISlm":"#2c6e49","VISa":"#8f5b0a"}
titles={"oddball":"Feature-oddball\ndeviant - equiprobable control","sequence":"Sequence\ndeviant - equiprobable control",
        "duration":"Duration / timing\nomission response at expected time"}
fig,axes=plt.subplots(1,3,figsize=(14,4.4)); fig.subplots_adjust(left=0.06,right=0.985,top=0.78,bottom=0.15,wspace=0.26)
for ax,para in zip(axes,["oddball","sequence","duration"]):
    meta,pe,TC=PE[para]; A=area_traces(meta,pe,TC); T=TC*1000
    for a in areas:
        if a in A:
            mu,se,nn=A[a]; mus=gaussian_filter1d(mu,0.7)
            ax.plot(T,mus,color=acol[a],lw=1.8,label=f"{a} (n={nn})"); ax.fill_between(T,mus-se,mus+se,color=acol[a],alpha=0.15,lw=0)
    ax.axvline(0,color="k",lw=0.5,ls=":"); ax.axhline(0,color="k",lw=0.6); ax.axvspan(0,150,color="#fdf0d5",zorder=0)
    ax.set_xlim(-100,400); ax.set_xlabel("time from (expected) onset (ms)"); ax.set_title(titles[para],loc="left",fontsize=7.6)
    ax.legend(frameon=False,fontsize=6.3,loc="upper right")
axes[0].set_ylabel("prediction-error signal (Hz)")
fig.suptitle("PE time courses by visual area — same animals across three error types",fontsize=9.2,y=0.95)
plt.show()

In [ ]:
# H1/H0 test: laminar profile + area x layer cross-correlation
def coarse(l):
    l=str(l)
    if l in ("1","2","2/3","3"): return "L1-3"
    if l=="4": return "L4"
    if l=="5": return "L5"
    if l.startswith("6"): return "L6"
    return "?"
def unit_mag(pe,TC,w=(0,0.15)):
    wm=(TC>=w[0])&(TC<=w[1]); return np.nanmean(pe[:,wm],1)
# laminar (collapse areas) and area x layer profiles
lay_prof={}; ax_prof={}
for para,(meta,pe,TC) in PE.items():
    m=meta.copy(); m["mag"]=unit_mag(pe,TC); m["lay"]=m.layer.map(coarse); m=m[m.lay!="?"]
    lp=m.groupby("lay").mag.agg(["median","count"]); lay_prof[para]=lp["median"].reindex(["L1-3","L4","L5","L6"])
    ap=m.groupby(["area","lay"]).mag.agg(["median","count"]); ax_prof[para]=ap["median"][ap["count"]>=8]
L=pd.DataFrame(lay_prof).reindex(["L1-3","L4","L5","L6"])
P=pd.DataFrame(ax_prof).dropna()
print("Laminar PE profile (median Hz):"); print(L.round(2).to_string())
print(f"\nArea x layer profiles: {len(P)} cells common to all three")
for a,b in [("oddball","sequence"),("oddball","duration"),("sequence","duration")]:
    if a in P and b in P:
        rho,p=spearmanr(P[a],P[b]); print(f"  {a} vs {b}: Spearman rho={rho:+.2f} (p={p:.3f})")

pcol={"oddball":"#1b4079","sequence":"#2c6e49","duration":"#8f5b0a"}
plab={"oddball":"Feature-oddball","sequence":"Sequence","duration":"Duration/timing"}
fig=plt.figure(figsize=(13,4.6)); gs=fig.add_gridspec(1,3,width_ratios=[1.15,1.15,1.0],left=0.06,right=0.985,top=0.82,bottom=0.14,wspace=0.34)
axA=fig.add_subplot(gs[0,0]); xx=np.arange(4)
for para in ["oddball","sequence","duration"]:
    v=L[para].values; vn=v/np.nanmax(v); axA.plot(xx,vn,"o-",color=pcol[para],lw=1.8,ms=6,label=plab[para])
axA.set_xticks(xx); axA.set_xticklabels(["L1-3","L4","L5","L6"]); axA.set_ylim(-0.05,1.32)
axA.set_ylabel("PE magnitude (normalized to each type's max)"); axA.set_xlabel("cortical layer")
axA.legend(frameon=False,fontsize=6.6,loc="upper left")
axA.set_title("A . Laminar profile - PE concentrates in L4-L6",loc="left",fontsize=7.8)
axB=fig.add_subplot(gs[0,1])
for para,mk in [("sequence","o"),("duration","s")]:
    if para in P: axB.scatter(P["oddball"],P[para],c=pcol[para],s=40,marker=mk,label=f"vs {plab[para]}",edgecolor="w",linewidth=0.5,zorder=3)
axB.axhline(0,color="k",lw=0.5); axB.axvline(0,color="k",lw=0.5); axB.plot([-0.3,5],[-0.3,5],color="0.7",lw=0.8,ls="--")
axB.set_xlabel("feature-oddball PE (Hz, per area x layer)"); axB.set_ylabel("other error type PE (Hz)"); axB.legend(frameon=False,fontsize=6.6,loc="upper left")
axB.set_title("B . Anatomical alignment across types",loc="left",fontsize=7.8)
axC=fig.add_subplot(gs[0,2]); pairs=[("oddball","sequence"),("oddball","duration"),("sequence","duration")]
rhos=[spearmanr(P[a],P[b])[0] if (a in P and b in P) else np.nan for a,b in pairs]
axC.bar(range(3),rhos,0.62,color=["#4a4a4a","#4a4a4a","#9a9a9a"]); axC.axhline(0,color="k",lw=0.8)
axC.set_xticks(range(3)); axC.set_xticklabels(["odd x seq","odd x dur","seq x dur"],fontsize=6.6); axC.set_ylabel("Spearman rho")
for i,rr in enumerate(rhos):
    if not np.isnan(rr): axC.text(i,rr+0.03,f"{rr:+.2f}",ha="center",fontsize=7)
axC.set_title("C . Profiles lean the same way (partial)",loc="left",fontsize=7.8)
fig.suptitle("H1 (shared substrate) vs H0 (distinct circuits): same anatomical populations across error types?",fontsize=9,y=0.955)
plt.show()

### Takeaway

- **One robust finding — a shared coarse laminar signature.** All three error types concentrate
  the prediction error in the **granular/infragranular layers (L4–L6)**, superficial L1–3 weakest.
  A common laminar bias independent of *how* the expectation was violated is what H1 predicts.
- **Partial, not conclusive.** At full area×layer resolution the profiles only *lean* the same way
  (feature-oddball vs sequence ρ ≈ +0.5, vs duration ρ ≈ +0.5, both n.s. at ~10 cells; sequence vs
  duration ≈ 0). The peak layer differs across types. This is **partial support for H1**, limited by
  power (5 animals, separate sessions per paradigm) and by the fact that we compare *populations*,
  not the *same units* across paradigms.
- **What would settle it:** the same units recorded across paradigms in one session, or many more
  CCF sessions to power the area×layer correlation.